# Tutorial: Colab Drive OOF Blend Pipeline

This notebook is the full working Colab pipeline for the repository.

It does four things:
- mounts Google Drive
- clones or updates the public GitHub repository
- runs `oof_blend_freq_sev` stability checks with checkpoint/resume
- runs the full pipeline and writes all outputs to Google Drive

Expected training data path:
- `/content/drive/MyDrive/risk_case/data/train.csv`

Expected output root:
- `/content/drive/MyDrive/risk_case/artifacts`


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tech-Alash/risk-case-championship.git"
REPO_DIR = Path("/content/risk-case-championship")
DRIVE_ROOT = Path("/content/drive/MyDrive/risk_case")
TRAIN_PATH = DRIVE_ROOT / "data" / "train.csv"
ARTIFACTS_ROOT = DRIVE_ROOT / "artifacts"
CONFIG_PATH = REPO_DIR / "configs" / "colab_drive_oof_blend.json"
SPLITS_PATH = REPO_DIR / "configs" / "experiments" / "stability_splits.json"

print({
    "python": sys.version.split()[0],
    "repo_url": REPO_URL,
    "repo_dir": str(REPO_DIR),
    "train_path": str(TRAIN_PATH),
    "artifacts_root": str(ARTIFACTS_ROOT),
})


In [ ]:
# Colab only: mount Google Drive.

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not running in Colab; skip drive mount.")


In [ ]:
# Clone the public repo on first run, otherwise fast-forward pull.

if REPO_DIR.exists():
    print("Repository already exists, pulling latest main...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("cwd =", Path.cwd())


In [ ]:
# Install dependencies into the current kernel environment.

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
# Quick environment check.

gpu_info = subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True)
print("HAS_GPU =", gpu_info.returncode == 0)
print(gpu_info.stdout[:1500] if gpu_info.returncode == 0 else gpu_info.stderr[:500])

assert TRAIN_PATH.exists(), f"Train file not found: {TRAIN_PATH}"
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
assert CONFIG_PATH.exists(), f"Config file not found: {CONFIG_PATH}"
assert SPLITS_PATH.exists(), f"Splits config not found: {SPLITS_PATH}"

cfg = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
print(json.dumps(cfg["paths"], ensure_ascii=False, indent=2))


## Stability Run

This is the main `oof_blend_freq_sev` stability run.

Outputs go to:
- `/content/drive/MyDrive/risk_case/artifacts/stability/<timestamp>/`

Checkpoint is saved automatically inside that timestamped folder.


In [ ]:
stability_cmd = [
    sys.executable,
    "scripts/run_stability_checks.py",
    "--config", str(CONFIG_PATH),
    "--splits_config", str(SPLITS_PATH),
    "--candidate", "oof_blend_freq_sev",
]

print("Running:")
print(" ".join(stability_cmd))
subprocess.run(stability_cmd, check=True)


## Resume From Latest Checkpoint

Use this if the Colab session disconnects before the stability run finishes.


In [ ]:
def latest_oof_checkpoint(artifacts_root: Path) -> Path:
    stability_root = artifacts_root / "stability"
    if not stability_root.exists():
        raise FileNotFoundError(f"No stability directory found: {stability_root}")
    checkpoints = sorted(
        stability_root.glob("*/oof_blend_checkpoint"),
        key=lambda path: path.parent.stat().st_mtime,
        reverse=True,
    )
    if not checkpoints:
        raise FileNotFoundError("No oof_blend checkpoint directory found under stability outputs")
    return checkpoints[0]

checkpoint_dir = latest_oof_checkpoint(ARTIFACTS_ROOT)
print("Latest checkpoint:", checkpoint_dir)

resume_cmd = [
    sys.executable,
    "scripts/run_stability_checks.py",
    "--config", str(CONFIG_PATH),
    "--splits_config", str(SPLITS_PATH),
    "--candidate", "oof_blend_freq_sev",
    "--resume_from", str(checkpoint_dir),
]

print("Running:")
print(" ".join(resume_cmd))
subprocess.run(resume_cmd, check=True)


## Full Pipeline Run

This trains and evaluates the full `oof_blend_freq_sev` pipeline and writes the run under:
- `/content/drive/MyDrive/risk_case/artifacts/runs/<run_id>/`

Because `test_csv` is `null` in this config, no submission file is generated.


In [ ]:
pipeline_cmd = [
    sys.executable,
    "scripts/run_pipeline.py",
    "--config", str(CONFIG_PATH),
]

print("Running:")
print(" ".join(pipeline_cmd))
subprocess.run(pipeline_cmd, check=True)


## Inspect Latest Outputs

This cell prints the latest stability summary and the latest full-run pointer.


In [ ]:
stability_root = ARTIFACTS_ROOT / "stability"
latest_stability_dirs = sorted(
    [path for path in stability_root.iterdir() if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if latest_stability_dirs:
    latest_stability = latest_stability_dirs[0]
    print("Latest stability dir:", latest_stability)
    summary_path = latest_stability / "summary.json"
    if summary_path.exists():
        print(summary_path.read_text(encoding="utf-8")[:4000])
    else:
        print("summary.json not found yet")
else:
    print("No stability outputs yet")

latest_run_path = ARTIFACTS_ROOT / "latest_run.json"
if latest_run_path.exists():
    latest_run = json.loads(latest_run_path.read_text(encoding="utf-8"))
    print(json.dumps(latest_run, ensure_ascii=False, indent=2))
else:
    print("latest_run.json not found yet")
